In [1]:
import json 
import os
import requests
from datetime import datetime, timedelta

In [2]:
def get_data(api_url):
    try:
        response = requests.get(
            api_url,
            headers={'Content-Type': 'application/json'}
        )
        
        response.raise_for_status()

        return response.json()
    except requests.exceptions.RequestException as e:
        print(f"Error making request: {e}")
        return None

def append_to_json_file(data, file):
    if os.path.exists(file):
        return 
    
    with open(file, 'w', encoding='utf-8') as json_file:
        json.dump(data, json_file, indent=4, ensure_ascii=False)
    print(f"Data saved to {file}")


def fetch_and_concatenate_data(apis):
    all_data = []

    for api_url in apis:
        data = get_data(api_url)

        if data:
            inhouse_reservations = [reservation for reservation in data 
                                   if reservation.get("reservationStatus") == "INHOUSE"]
            all_data.extend(inhouse_reservations)
            print(f"Fetched {len(data)} reservations, kept {len(inhouse_reservations)} with INHOUSE status from {api_url}")

        else:
            print(f"Failed to fetch data from {api_url}")

    return all_data


def load_json_from_file(file_path):
    if not os.path.exists(file_path):
        print(f"Error: File not found at {file_path}")
        return None
    
    try:
        with open(file_path, 'r', encoding='utf-8') as json_file:
            data = json.load(json_file)
        return data
    except json.JSONDecodeError:
        print(f"Error: Could not decode JSON from {file_path}. The file might be corrupted or not in valid JSON format.")
        return None
    except Exception as e:
        print(f"An unexpected error occurred while reading {file_path}: {e}")
        return None

### Etract and save The PMS data into a json file 

In [ ]:
def fetch_and_save_data_in_chunks(overall_start_date_str, overall_end_date_str, output_json_file_path, api_templates=None):
    """
    Fetches reservation data in 4-day (or smaller for the last chunk) intervals
    for a given date range and appends each chunk's data to a single JSON file.
    The JSON file will store a list of all reservations.
    """
    try:
        overall_start_date_obj = datetime.strptime(overall_start_date_str, "%Y-%m-%d")
        overall_end_date_obj = datetime.strptime(overall_end_date_str, "%Y-%m-%d")
    except ValueError:
        print("Error: Invalid date format. Please use YYYY-MM-DD.")
        return

    if overall_start_date_obj > overall_end_date_obj:
        print("Error: Start date cannot be after end date.")
        return

    all_reservations = []
    if os.path.exists(output_json_file_path):
    
        try:
            with open(output_json_file_path, 'r', encoding='utf-8') as f:
                content = f.read()
    
                if content.strip():  # Check if file is not empty
                    existing_data = json.loads(content)
    
                    if isinstance(existing_data, list):
                        all_reservations = existing_data
                        print(f"Loaded {len(all_reservations)} existing records from {output_json_file_path}")
    
                    else:
                        print(f"Warning: Existing content in {output_json_file_path} is not a list. Starting with an empty list.")
                        all_reservations = [] # Ensure it's a list
    
                else:
                    print(f"{output_json_file_path} is empty. Starting with an empty list.")
    
        except json.JSONDecodeError:
            print(f"Warning: Could not decode JSON from {output_json_file_path}. Starting with an empty list.")
    
        except Exception as e:
            print(f"An unexpected error occurred while reading {output_json_file_path}: {e}. Starting with an empty list.")
    
    else:
        print(f"{output_json_file_path} not found. A new file will be created.")

    current_chunk_start_date_obj = overall_start_date_obj
    total_new_records_fetched_this_run = 0

    while current_chunk_start_date_obj <= overall_end_date_obj:
        chunk_period_end_date_obj = current_chunk_start_date_obj + timedelta(days=3)

        if chunk_period_end_date_obj > overall_end_date_obj:
            chunk_period_end_date_obj = overall_end_date_obj

        chunk_start_str = current_chunk_start_date_obj.strftime("%Y-%m-%d")
        chunk_end_str = chunk_period_end_date_obj.strftime("%Y-%m-%d")

        print(f"\nProcessing chunk: {chunk_start_str} to {chunk_end_str}")

        apis_for_chunk = [template.format(chunk_start_str, chunk_end_str) for template in api_templates]
        # print(f"Constructed {len(apis_for_chunk)} API URLs for the current chunk.")

        chunk_data = fetch_and_concatenate_data(apis_for_chunk)

        if chunk_data:
            all_reservations.extend(chunk_data)
            total_new_records_fetched_this_run += len(chunk_data)
            
            try:
                # Ensure the directory for the output file exists
                output_dir = os.path.dirname(output_json_file_path)
            
                if output_dir and not os.path.exists(output_dir):
                    os.makedirs(output_dir)
                    print(f"Created directory: {output_dir}")
                
                with open(output_json_file_path, 'w', encoding='utf-8') as json_file:
                    json.dump(all_reservations, json_file, indent=4, ensure_ascii=False)
                print(f"Data for chunk {chunk_start_str} to {chunk_end_str} processed. Total records in {output_json_file_path}: {len(all_reservations)}")
            
            except IOError as e:
                print(f"Error writing data to {output_json_file_path} for chunk {chunk_start_str}-{chunk_end_str}: {e}")
            
            except TypeError as e:
                print(f"Error preparing data for JSON serialization for chunk {chunk_start_str}-{chunk_end_str}: {e}")
        
        else:
            print(f"No new data fetched or an error occurred for chunk {chunk_start_str} to {chunk_end_str}.")

        current_chunk_start_date_obj = chunk_period_end_date_obj + timedelta(days=1)

    print(f"\nFinished processing all chunks. Total new records fetched in this run: {total_new_records_fetched_this_run}.")
    print(f"Final total records in {output_json_file_path}: {len(all_reservations)}")

In [4]:
api_templates = [
    'https://pmsvaleriaapi.fractalstay.com/api/reservations?&from={}&to={}&group=1&resastatus=0',
    'https://pmsvaleriaapi.fractalstay.com/api/reservations?&from={}&to={}&group=1&resastatus=3',
    'https://pmsvaleriaapi.fractalstay.com/api/reservations?&from={}&to={}&group=1&resastatus=2',
]

start_date_range = "2024-01-01"
end_date_range = "2025-12-30" 
output_path = "../Data/PMSEtractedData/PMSreservations.json" 

fetch_and_save_data_in_chunks(start_date_range, end_date_range, output_path, api_templates)

../Data/PMSEtractedData/PMSreservations.json not found. A new file will be created.

Processing chunk: 2024-01-01 to 2024-01-04
Failed to fetch data from https://pmsvaleriaapi.fractalstay.com/api/reservations?&from=2024-01-01&to=2024-01-04&group=1&resastatus=0
Failed to fetch data from https://pmsvaleriaapi.fractalstay.com/api/reservations?&from=2024-01-01&to=2024-01-04&group=1&resastatus=3
Failed to fetch data from https://pmsvaleriaapi.fractalstay.com/api/reservations?&from=2024-01-01&to=2024-01-04&group=1&resastatus=2
No new data fetched or an error occurred for chunk 2024-01-01 to 2024-01-04.

Processing chunk: 2024-01-05 to 2024-01-08
Failed to fetch data from https://pmsvaleriaapi.fractalstay.com/api/reservations?&from=2024-01-05&to=2024-01-08&group=1&resastatus=0
Failed to fetch data from https://pmsvaleriaapi.fractalstay.com/api/reservations?&from=2024-01-05&to=2024-01-08&group=1&resastatus=3
Failed to fetch data from https://pmsvaleriaapi.fractalstay.com/api/reservations?&from